Nairobi UXsim — Methodology v2
=====================================================================
Traffic simulation of Nairobi's strategic road network to rank urban-planning interventions by travel-time impact.

Sources synthesised in this notebook:
- **NIUPLAN** (JICA, 2014): missing-link register (ML-01..ML-16), BRT routes 1–6, LRT assignments, phasing, baseline speeds/volumes
- **NaMSIP Eastlands Urban Renewal Plan** (Real Plan Consultants, 2019): Nairobi Riverfront corridor, Eastleigh/Jogoo widenings, junction programme
- **Njeru (2024)**: NIUPLAN implementation-gap analysis (motivation: only 1 of 38 priority projects completed)

v2 methodology fixes (vs first version):
1. Aborted trips (`trip_abort==1`) are excluded from average travel time and reported separately
2. Demand is loaded in the first half of the horizon so trips can physically complete
3. Disruption capacity restoration is idempotent (no double-disruption corruption)
4. Common random numbers: every candidate faces identical stochastic disruptions; results averaged over replications
5. Congestion level is calibrated to NIUPLAN's ~20 km/h do-nothing network speed via a link-capacity factor (no more blanket 11 km/h cap)
6. Junction indiscipline constrains node flow capacity, not link capacity
7. BRT/LRT candidates include a modal-relief proxy (share of corridor car demand removed)
8. Greedy search is parallelised and followed by a one-pass swap test


In [ ]:
# %SMOKE:skip
!pip install -q "uxsim>=1.5" osmnx requests

import os, sys, time, json, hashlib, random, warnings
from collections import Counter
from concurrent.futures import ProcessPoolExecutor
import multiprocessing as mp

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

from uxsim import World
from uxsim.OSMImporter import OSMImporter

import uxsim
print("Setup complete | uxsim", getattr(uxsim, "__version__", "?"), "| python", sys.version.split()[0])


In [ ]:
# %SMOKE:config
CONFIG = {
    # network
    'bbox':       (36.65, -1.38, 36.95, -1.15),
    'osm_filter': '["highway"~"trunk|primary|secondary"]',
    'node_merge_threshold':   0.003,
    'node_merge_iterations':  2,
    # simulation
    'deltan':     10,
    'tmax':       10800,          # 3 h horizon
    'demand_window_frac': 0.5,    # departures in [0, tmax*frac] so trips can finish (fix v1 artifact)
    'random_seed': 42,
    'speed_cap_ms': 22.2,         # 80 km/h absolute ceiling only; OSM speeds kept otherwise
    'capacity_factor_default': 0.7,
    'target_network_speed_kmh': 20.0,  # NIUPLAN do-nothing 2030 city average (~20.2 km/h)
    # experiment
    'n_replicates_search': 2,
    'n_replicates_final':  3,
    'n_jobs': max(1, min(4, os.cpu_count() or 1)),
    # io
    'results_dir': '/kaggle/working/results' if os.path.exists('/kaggle/working') else 'results',
}
CONFIG['capacity_factor'] = None   # set by calibrate_capacity()

os.makedirs(CONFIG['results_dir'], exist_ok=True)
os.makedirs(f"{CONFIG['results_dir']}/interventions", exist_ok=True)
print(f"Config loaded | n_jobs={CONFIG['n_jobs']} | results={CONFIG['results_dir']}")


In [ ]:
# %SMOKE:skip
try:
    r = requests.get("https://httpbin.org/ip", timeout=5)
    print("Internet ON, IP:", r.json()["origin"])
except Exception as e:
    print("Internet OFF or blocked:", e)


In [ ]:
# %SMOKE:skip
def build_network(force_download=False):
    """Download/process OSM once, then reuse CSV cache across sessions."""
    cache_nodes = f"{CONFIG['results_dir']}/net_cache_nodes.csv"
    cache_links = f"{CONFIG['results_dir']}/net_cache_links.csv"
    if not force_download and os.path.exists(cache_nodes) and os.path.exists(cache_links):
        nodes = pd.read_csv(cache_nodes)
        links = pd.read_csv(cache_links)
        print(f"Loaded cached network: {len(links)} links, {len(nodes)} nodes")
        return nodes, links

    print("Downloading Nairobi network...")
    start = time.time()
    nodes, links = OSMImporter.import_osm_data(
        bbox=CONFIG['bbox'],
        custom_filter=CONFIG['osm_filter'],
    )
    print(f"Raw: {len(links)} links, {len(nodes)} nodes")
    nodes, links = OSMImporter.osm_network_postprocessing(
        nodes, links,
        node_merge_threshold=CONFIG['node_merge_threshold'],
        node_merge_iteration=CONFIG['node_merge_iterations'],
        enforce_bidirectional=True,
    )
    print(f"Processed: {len(links)} links, {len(nodes)} nodes | {time.time()-start:.1f}s")
    try:
        nodes.to_csv(cache_nodes, index=False)
        links.to_csv(cache_links, index=False)
        print("Network cached to CSV")
    except Exception as e:
        print(f"(cache write skipped: {e})")
    return nodes, links

_nodes, _links = build_network()

OSMImporter.osm_network_visualize(_nodes, _links, show_link_name=0)
plt.title("Nairobi Road Network — UXsim")
plt.show()


In [ ]:
# %SMOKE:interventions
def link_matches(link, pattern):
    """Match a UXsim link against a road-name pattern.
    OSMImporter names look like 'Jogoo Road-12345' or 'Jogoo Road-12345-reverse',
    so we compare against both the bare road name and the full link name."""
    p = pattern.strip().lower()
    full = str(link.name).lower()
    road = full.split('-')[0].strip()
    return (p in road) or (p in full)


class UpgradeLink:
    """Upgrade existing links matching name patterns (widening / resurfacing / BRT infra).
    relief: optional [(pattern, frac)] — share of car demand removed on matching OD
    corridors as a modal-shift proxy for transit interventions (NaMSIP assumes large
    bus->BRT conversion)."""
    def __init__(self, id, name, pattern, capacity_mult=1.5, speed_mult=1.2,
                 rationale="", relief=None):
        self.id = id
        self.name = name
        self.pattern = pattern
        self.capacity_mult = capacity_mult
        self.speed_mult = speed_mult
        self.rationale = rationale
        self.relief = list(relief or [])

    def apply(self, W):
        modified = 0
        for link in W.LINKS:
            if any(link_matches(link, p) for p in self.pattern.split('|')):
                link.capacity *= self.capacity_mult
                capped_speed = min(link.free_flow_speed * self.speed_mult, 16.7)  # <=60 km/h
                link.change_free_flow_speed(capped_speed)
                modified += 1
        if modified == 0:
            print(f"Warning: no links matched '{self.pattern}' ({self.id})")
        return modified


class AddLink:
    """Insert a new link between the existing nodes nearest two coordinates."""
    def __init__(self, id, name, start_coords, end_coords, length_m,
                 speed_ms=11.1, lanes=2, rationale=""):
        self.id = id
        self.name = name
        self.start_coords = start_coords
        self.end_coords = end_coords
        self.length = length_m
        self.speed = speed_ms
        self.lanes = lanes
        self.rationale = rationale
        self.relief = []

    def apply(self, W):
        start_node = end_node = None
        best_s = best_e = float('inf')
        sx, sy = self.start_coords
        ex, ey = self.end_coords
        for node in W.NODES:                      # single pass for both endpoints
            ds = (node.x - sx)**2 + (node.y - sy)**2
            if ds < best_s:
                best_s, start_node = ds, node
            de = (node.x - ex)**2 + (node.y - ey)**2
            if de < best_e:
                best_e, end_node = de, node
        if start_node is None or end_node is None:
            print(f"Warning: could not anchor {self.name}")
            return 0
        if any(str(l.name) == self.name for l in W.LINKS):
            print(f"Warning: link '{self.name}' already exists, skipping")
            return 0
        try:
            W.addLink(name=self.name, start_node=start_node.name, end_node=end_node.name,
                      length=self.length, free_flow_speed=self.speed,
                      number_of_lanes=self.lanes)
            return 1
        except Exception as e:
            print(f"Warning: {self.name}: {e}")
            return 0


class InterventionGroup:
    """Bundle several primitive interventions into one searchable candidate."""
    def __init__(self, id, name, parts, rationale=""):
        self.id = id
        self.name = name
        self.parts = parts
        self.rationale = rationale
        self.relief = []
        for part in parts:
            self.relief.extend(getattr(part, 'relief', []))

    def apply(self, W):
        return sum(part.apply(W) for part in self.parts)


print("Intervention classes defined")


In [ ]:
# %SMOKE:demand
# Peak-hour corridor demand (volumes at scale=1.0 over the demand window)
BASE_DEMANDS = [
    (336.869, -1.219, 0.02, 36.820, -1.283, 0.02, 13000, "Thika Road"),
    (36.870, -1.340, 0.02, 36.820, -1.283, 0.02, 8000,  "Mombasa Road"),
    (36.740, -1.330, 0.02, 36.820, -1.283, 0.02, 7000,  "Ngong Road"),
    (36.730, -1.265, 0.02, 36.820, -1.283, 0.02, 8000,  "Waiyaki Way"),
    (36.870, -1.270, 0.02, 36.820, -1.283, 0.02, 11500, "Jogoo Road"),
    (36.808, -1.267, 0.015, 36.820, -1.300, 0.015, 4500, "Cross-city A"),
    (36.820, -1.259, 0.015, 36.860, -1.270, 0.015, 3500, "Cross-city B"),
    (36.755, -1.340, 0.015, 36.808, -1.267, 0.015, 3000, "Karen"),
    (36.820, -1.283, 0.02, 36.869, -1.219, 0.02, 2500,  "Rev Thika"),
    (36.820, -1.283, 0.02, 36.730, -1.265, 0.02, 2000,  "Rev Waiyaki"),
    (36.820, -1.283, 0.02, 36.740, -1.330, 0.02, 2000,  "Rev Ngong"),
]

def add_demand(W, scale=1.0, relief=None):
    """Load demand uniformly inside [0, tmax * demand_window_frac].
    relief maps OD-label patterns -> fraction of car demand removed (modal shift proxy)."""
    relief = relief or {}
    t_end = CONFIG['tmax'] * CONFIG['demand_window_frac']
    for ol, ola, orr, dl, dla, dr, vol, label in BASE_DEMANDS:
        mult = 1.0
        lbl = label.lower()
        for pat, frac in relief.items():
            if frac > 0 and pat.lower() in lbl:
                mult *= (1.0 - min(frac, 0.9))
        W.adddemand_area2area2(
            ol, ola, orr, dl, dla, dr,
            0, t_end,
            volume=int(vol * scale * mult),
        )

print(f"Base demand: {sum(d[6] for d in BASE_DEMANDS):,} vehicles at scale=1.0 "
      f"| window [0, {int(CONFIG['tmax']*CONFIG['demand_window_frac'])}s]")


In [ ]:
# %SMOKE:disruptions
import random

class NairobiDisruptions:
    """
    Nairobi-specific operational frictions:
      1. random matatu stops on matatu corridors (partial lane blockage)
      2. lane indiscipline at >=3-arm junctions (node flow-capacity reduction)
      3. random accidents blocking the most-loaded approach at blackspots
      4. spatial speed variation (CBD slower than periphery)

    Provenance (as coded in v1): Ma3Route/World Bank blackspot data,
    NUTRANS field observations, Digital Matatus route density.

    v2 changes:
      - uses an injected random.Random instance -> reproducible per scenario (CRN)
      - capacity restoration via original-capacity ledger: stacking blocks on the
        same link can no longer overwrite the true original capacity
    """

    MATATU_CORRIDORS = [
        'Thika Road', 'Jogoo', 'Mombasa Road', 'Ngong Road', 'Waiyaki',
        'Juja Road', 'Muranga', 'Langata', 'Outer Ring',
    ]
    BLACKSPOT_KEYWORDS = [
        'Westlands', 'Allsops', 'Roysambu', 'GPO', 'Globe',
        'Museum Hill', 'Community', 'Nyayo', 'Haile',
    ]

    def __init__(self,
                 matatu_stop_prob=0.003,
                 matatu_dwell_range=(30, 120),
                 accident_prob=0.001,
                 accident_duration_range=(120, 600),
                 indiscipline_factor=0.7,
                 rng=None):
        self.matatu_stop_prob    = matatu_stop_prob
        self.matatu_dwell_range  = matatu_dwell_range
        self.accident_prob       = accident_prob
        self.accident_duration   = accident_duration_range
        self.indiscipline_factor = indiscipline_factor
        self.rng = rng if rng is not None else random.Random()

        self._orig_cap = {}   # link -> capacity before any active disruption
        self._active   = []   # [(restore_time, link)]

    # ---- static -------------------------------------------------------
    def apply_static(self, W):
        """Junction indiscipline on node flow capacity + CBD speed penalty."""
        junctions = 0
        for node in W.NODES:
            if len(node.inlinks) >= 3:
                is_matatu = any(
                    link_matches(l, kw)
                    for l in node.inlinks.values()
                    for kw in self.MATATU_CORRIDORS
                )
                factor = self.indiscipline_factor - (0.1 if is_matatu else 0.0)
                base = node.flow_capacity
                if not base:  # None/0 -> derive from approaches
                    base = sum(l.capacity for l in node.inlinks.values())
                node.flow_capacity = base * factor
                junctions += 1

        CBD_LON, CBD_LAT = 36.8219, -1.2833
        speeds_reduced = 0
        for link in W.LINKS:
            dist = ((link.start_node.x - CBD_LON)**2 +
                    (link.start_node.y - CBD_LAT)**2)**0.5
            if dist < 0.02:               # ~2 km from CBD
                link.free_flow_speed *= 0.7
                speeds_reduced += 1

        print(f"  Static disruptions: indiscipline on {junctions} nodes, "
              f"CBD slowdown on {speeds_reduced} links")

    # ---- dynamic ------------------------------------------------------
    def step(self, W, current_time):
        self._restore_expired(current_time)

        for link in W.LINKS:
            if link.num_vehicles == 0:
                continue
            if not any(link_matches(link, kw) for kw in self.MATATU_CORRIDORS):
                continue
            if self.rng.random() < self.matatu_stop_prob:
                dwell = self.rng.randint(*self.matatu_dwell_range)
                self._block(link, self.rng.uniform(0.3, 0.6), current_time + dwell)

        for node in W.NODES:
            if not any(kw.lower() in str(node.name).lower()
                       for kw in self.BLACKSPOT_KEYWORDS):
                continue
            if self.rng.random() < self.accident_prob:
                inlinks = list(node.inlinks.values())
                if not inlinks:
                    continue
                worst = max(inlinks, key=lambda l: l.num_vehicles)
                duration = self.rng.randint(*self.accident_duration)
                self._block(worst, 0.1, current_time + duration)

    # ---- internals ----------------------------------------------------
    def _block(self, link, block_factor, until):
        if link not in self._orig_cap:
            self._orig_cap[link] = link.capacity
        link.capacity = self._orig_cap[link] * block_factor
        self._active.append((until, link))

    def _restore_expired(self, current_time):
        due = [e for e in self._active if current_time >= e[0]]
        self._active = [e for e in self._active if current_time < e[0]]
        for _, link in due:
            if link in self._orig_cap:          # skip stale entries from stacked blocks
                link.capacity = self._orig_cap.pop(link)


print("NairobiDisruptions defined (v2: seeded RNG + safe restore)")


In [ ]:
# %SMOKE:helpers
import hashlib

def stable_seed(*parts):
    """Deterministic seed independent of PYTHONHASHSEED (safe across worker processes)."""
    h = hashlib.sha256("|".join(map(str, parts)).encode()).digest()
    return int.from_bytes(h[:8], "little")


def classify_vehicles(W):
    """A1 fix: aborted trips (trip_abort==1) are their own class and never count
    as completed, even though UXsim gives them a positive travel_time."""
    done, aborted, running = [], [], []
    for v in W.VEHICLES.values():
        tt = getattr(v, 'travel_time', None)
        if getattr(v, 'trip_abort', 0) == 1:
            aborted.append(v)
        elif tt is not None and tt > 0:
            done.append(v)
        else:
            running.append(v)
    return done, aborted, running


def trip_metrics(W):
    done, aborted, running = classify_vehicles(W)
    tts = np.array([v.travel_time for v in done], dtype=float)
    total = len(done) + len(aborted) + len(running)
    if tts.size == 0:
        tts = np.array([0.0])
    return {
        'n_total': total,
        'n_completed': len(done),
        'n_aborted': len(aborted),
        'n_incomplete': len(running),
        'completion_rate': len(done) / max(total, 1),
        'abort_rate': len(aborted) / max(total, 1),
        'att_s': float(tts.mean()),
        'att_median_s': float(np.median(tts)),
        'att_p90_s': float(np.percentile(tts, 90)),
        'share_over_60min': float((tts > 3600).mean()),
        'total_vehicle_hours': float(tts.sum() / 3600.0),
    }


class NetworkMonitor:
    """Time-average network speed weighted by vehicles present, sampled every tick
    via the user_function hook. Used as the calibration observable."""
    def __init__(self):
        self._speed_sum = 0.0
        self._veh_sum = 0.0

    def sample(self, W):
        veh = sum(l.num_vehicles for l in W.LINKS)
        if veh > 0:
            self._speed_sum += sum((l.speed or 0.0) * l.num_vehicles for l in W.LINKS)
            self._veh_sum += veh

    def avg_kmh(self):
        if self._veh_sum == 0:
            return float('nan')
        return (self._speed_sum / self._veh_sum) * 3.6


print("Metrics helpers defined")


In [ ]:
# %SMOKE:skip
CAPACITY_FACTOR_STATE = {'value': None}

def _current_capacity_factor():
    return CAPACITY_FACTOR_STATE['value'] or CONFIG['capacity_factor_default']


def create_world(interventions=None, demand_scale=1.0, disruptions=True,
                 rep=0, capacity_factor=None):
    cf = capacity_factor if capacity_factor is not None else _current_capacity_factor()

    iv_key = tuple(iv.id for iv in (interventions or []))
    dis_rng = random.Random(stable_seed(CONFIG['random_seed'], iv_key, rep))  # A4 CRN

    W = World(
        name="nairobi",
        deltan=CONFIG['deltan'],
        tmax=CONFIG['tmax'],
        print_mode=0, save_mode=0, show_mode=0,
        random_seed=CONFIG['random_seed'],
    )
    OSMImporter.osm_network_to_World(
        W, _nodes, _links,
        default_jam_density=0.2,
        coef_degree_to_meter=111000,
    )

    capped = 0
    for link in W.LINKS:
        if link.free_flow_speed > CONFIG['speed_cap_ms']:
            link.change_free_flow_speed(CONFIG['speed_cap_ms'])
            capped += 1
        link.capacity *= cf   # B5: calibration lever replaces old 0.3 blanket cut
    print(f"  cf={cf:.2f} | speed-capped links: {capped}")

    disrupt = None
    if disruptions:
        disrupt = NairobiDisruptions(rng=dis_rng)
        disrupt.apply_static(W)

    if interventions:
        for iv in interventions:
            iv.apply(W)

    relief = {}
    if interventions:
        for iv in interventions:
            for pat, frac in getattr(iv, 'relief', []):
                relief[pat] = min(relief.get(pat, 0.0) + frac, 0.9)

    add_demand(W, scale=demand_scale, relief=relief)

    monitor = NetworkMonitor()

    def _tick(W_cb):
        if disrupt is not None:
            disrupt.step(W_cb, W_cb.T)
        monitor.sample(W_cb)

    W.user_function = _tick
    return W, monitor


def run_sim(interventions=None, scale=1.0, disruptions=True,
            replicates=None, capacity_factor=None):
    """Run one or more replications; aggregate metrics across them.
    Returns (last_world, aggregated_metrics_dict)."""
    if replicates is None:
        replicates = CONFIG['n_replicates_final']
    metrics, last_W, speeds = [], None, []
    for rep in range(replicates):
        W, mon = create_world(interventions, demand_scale=scale,
                              disruptions=disruptions, rep=rep,
                              capacity_factor=capacity_factor)
        W.exec_simulation()
        metrics.append(trip_metrics(W))
        speeds.append(mon.avg_kmh())
        last_W = W
    agg = {k: float(np.mean([m[k] for m in metrics])) for k in metrics[0]}
    agg['net_speed_kmh'] = float(np.nanmean(speeds))
    agg['n_replicates'] = replicates
    return last_W, agg


def calibrate_capacity(target_kmh=None):
    """Pick link-capacity factor whose simulated network speed matches NIUPLAN's
    do-nothing congested average (~20 km/h). Replaces v1's circular ATT-threshold scan."""
    target = target_kmh or CONFIG['target_network_speed_kmh']
    print(f"Calibrating capacity factor to network speed ~= {target} km/h ...")
    best = None
    for cf in [1.0, 0.85, 0.7, 0.55, 0.4, 0.3]:
        _, m = run_sim(scale=1.0, disruptions=True, replicates=1, capacity_factor=cf)
        spd, comp, att = m['net_speed_kmh'], m['completion_rate'], m['att_s'] / 60
        err = abs(spd - target)
        penalty = err + (100.0 if comp < 0.55 else 0.0)   # avoid collapsed-network fits
        print(f"  cf={cf:.2f} -> speed={spd:5.1f} km/h | completion={comp*100:4.0f}% "
              f"| ATT={att:5.1f} min | err={err:4.1f}")
        if best is None or penalty < best[0]:
            best = (penalty, cf, m)
    CAPACITY_FACTOR_STATE['value'] = best[1]
    print(f"Calibrated capacity_factor={best[1]:.2f}")
    with open(f"{CONFIG['results_dir']}/calibration.json", 'w') as f:
        json.dump({'capacity_factor': best[1],
                   'target_speed_kmh': target,
                   'achieved_speed_kmh': best[2]['net_speed_kmh'],
                   'completion_rate': best[2]['completion_rate']}, f, indent=2)
    return best[1]


print("World factory, run_sim and calibrate_capacity defined")


In [ ]:
# %SMOKE:skip
# Sanity: does every demand zone actually land on network nodes?
W_test = World(name="test", deltan=5, tmax=100,
               print_mode=0, save_mode=0, show_mode=0, random_seed=42)
OSMImporter.osm_network_to_World(W_test, _nodes, _links,
                                 default_jam_density=0.2,
                                 coef_degree_to_meter=111000)

print(f"{'Corridor':<15} {'Vehicles':>9}")
print("-" * 26)
for ol, ola, orr, dl, dla, dr, vol, label in BASE_DEMANDS:
    before = len(W_test.VEHICLES)
    W_test.adddemand_area2area2(ol, ola, orr, dl, dla, dr, 0, 100, volume=100)
    print(f"{label:<15} {len(W_test.VEHICLES)-before:>9}")
print("(counts are platoon multiples of deltan=5 — expected)")


In [ ]:
# %SMOKE:skip
CAPACITY_FACTOR = calibrate_capacity()


In [ ]:
# %SMOKE:skip
# Validate the model against projects that were ACTUALLY built (NIUPLAN short-term
# list): Enterprise Rd widening ($15M, completed per Njeru Table 9) and Outer Ring
# Rd dualing (KeNHA/KURA, completed 2016-ish). Direction of effect should be <= 0.
VAL_ENT = UpgradeLink(
    id='VAL_ENT', name='Enterprise Road widening (built)',
    pattern='Enterprise', capacity_mult=2.0, speed_mult=1.15,
    rationale='Validation: NIUPLAN short-term project, implemented.')
VAL_ORW = UpgradeLink(
    id='VAL_ORW', name='Outer Ring dual carriageway (built)',
    pattern='Outer Ring', capacity_mult=2.0, speed_mult=1.25,
    rationale='Validation: corridor already upgraded in reality.')

_, base_m = run_sim(scale=1.0, replicates=1)
for val in (VAL_ENT, VAL_ORW):
    _, m = run_sim(interventions=[val], scale=1.0, replicates=1)
    d = (base_m['att_s'] - m['att_s']) / base_m['att_s'] * 100
    verdict = "OK" if d > 0 else "WARN: built project shows no gain"
    print(f"{val.id}: ATT {base_m['att_s']/60:.1f} -> {m['att_s']/60:.1f} min "
          f"({d:+.1f}%)  [{verdict}]")


In [ ]:
# %SMOKE:skip
# Candidate interventions. Sources cited per item:
#   [NIUPLAN] JICA 2014 final report (missing-link register & Ch.7 transport plan)
#   [NaMSIP]  Eastlands Urban Renewal Plan Vol.2, 2019
#   [Gap]     network-gap inference (not a numbered plan link) — flagged honestly

RIVERFRONT_SEGMENTS = [
    AddLink('NRR_01a', 'Riverfront Road — Quarry to Gikomba',
            (36.8005, -1.2953), (36.8029, -1.2987), 800, 8.3, 2,
            'Segment 1 (anchor node 292006372 -> 81356873)'),
    AddLink('NRR_01b', 'Riverfront Road — Gikomba to Pumwani',
            (36.8029, -1.2987), (36.8292, -1.2979), 2800, 8.3, 2,
            'Segment 2 (81356873 -> 30498657)'),
    AddLink('NRR_01c', 'Riverfront Road — Pumwani to Eastleigh 1st Ave',
            (36.8292, -1.2979), (36.8390, -1.3041), 1200, 8.3, 2,
            'Segment 3 (30498657 -> 268651487)'),
    AddLink('NRR_01d', 'Riverfront Road — Eastleigh to Rabai Road',
            (36.8390, -1.3041), (36.8504, -1.2791), 3100, 8.3, 2,
            'Segment 4 (268651487 -> Rabai anchor)'),
]

INTERVENTIONS = [
    # ---- missing links [NIUPLAN register] ---------------------------------
    UpgradeLink('ML_05', 'General Waruinge St — Muratina to Juja Rd',
                'Waruinge|Muratina', 1.4, 1.2,
                'Missing Link No. 5, 3.0 km (EU/KURA package).'),
    UpgradeLink('ML_10', 'Likoni Road extension corridor',
                'Likoni', 1.4, 1.15,
                'Missing Link No. 10: Enterprise Rd to Mombasa Rd, 1.8 km.'),
    UpgradeLink('ML_15B', 'Ring Road Parklands extension',
                'Ring Road Parklands|Parklands', 1.5, 1.2,
                'Missing Link No. 15b: Limuru Rd to Thika Superhighway, 1.6 km.'),
    UpgradeLink('ML_01', 'Accra Road — Ngara Road link',
                'Accra|Ngara', 1.4, 1.15,
                'Missing Link No. 1: Accra Rd to Ngara Rd, 0.7 km. '
                '(v1 mislabeled this ML_16.)'),
    AddLink('QRY_EXT', 'Quarry Road extension (Landhies — Quarry)',
            (36.8148, -1.2817), (36.8005, -1.2953), 2500, 9.0, 2,
            'Missing Link No. 16: Landhies Rd to Quarry Rd, 2.5 km [NIUPLAN].'),
    AddLink('NET_JOR', 'Jogoo — Outer Ring connector',
            (36.8620, -1.2905), (36.8767, -1.2970), 1800, 10.0, 2,
            '[Gap] v1 ML_13; not a numbered NIUPLAN link — models a missing '
            'east-west connector in the Jogoo/Outering grid.'),
    UpgradeLink('NET_MIR', 'Mombasa Rd industrial-area relief',
                'Enterprise|Lusaka', 1.5, 1.15,
                '[Gap] v1 ML_14; proxies industrial-area access relief via '
                'Enterprise/Lusaka upgrades (both NIUPLAN widening items).'),

    # ---- transit (with modal-relief proxy) --------------------------------
    UpgradeLink('BRT_L2', 'BRT Simba — Thika Road', 'Thika',
                1.35, 1.10, 'NIUPLAN BRT Route 1 (NAMATA Line 2 naming).',
                relief=[('thika', 0.15)]),
    UpgradeLink('BRT_L3', 'BRT Chui — Juja Road', 'Juja',
                1.35, 1.10, 'NIUPLAN BRT Route 2.',
                relief=[('juja', 0.12)]),
    UpgradeLink('BRT_L1', 'BRT Ndovu — Mombasa Road', 'Mombasa',
                1.35, 1.10, 'NIUPLAN BRT Route 3.',
                relief=[('mombasa', 0.12)]),
    UpgradeLink('BRT_L4', 'Jogoo Road rapid transit', 'Jogoo',
                1.35, 1.10,
                'NIUPLAN assigns Jogoo an LRT long-term; modelled as transit '
                'upgrade + modal relief.',
                relief=[('jogoo', 0.18)]),
    UpgradeLink('BRT_L5', 'BRT Nyati — Outer Ring Road', 'Outer Ring',
                1.35, 1.10, 'NIUPLAN BRT Route 6.',
                relief=[('outer ring', 0.15)]),
    UpgradeLink('BRT_WKY', 'Waiyaki Way rapid transit', 'Waiyaki',
                1.35, 1.10,
                'NIUPLAN Route 4 — selected mode is BRT (v1 wrongly used LRT).',
                relief=[('waiyaki', 0.12)]),
    UpgradeLink('LRT_OR', 'Outer Ring LRT upgrade', 'Outer Ring',
                1.30, 1.10,
                'NIUPLAN: LRT after dualling, long-term.',
                relief=[('outer ring', 0.20)]),

    # ---- highway upgrades --------------------------------------------------
    UpgradeLink('HWY_02', 'Mombasa Road capacity enhancement', 'Mombasa',
                1.6, 1.2,
                'NIUPLAN medium/long-term widening JKIA—James Gichuru / Athi River.'),
    UpgradeLink('HWY_03', 'Outer Ring Road upgrade (dual carriageway)', 'Outer Ring',
                1.6, 1.25,
                'NIUPLAN short-term widening; built in reality (also used for '
                'validation as VAL_ORW).'),
    UpgradeLink('JLR_01', 'Jogoo–Landhies widening', 'Jogoo|Landhies',
                1.4, 1.15,
                '[NaMSIP] Jogoo reserve 55→60 m; Landhies 35→40 m.'),

    # ---- Eastlands [NaMSIP] -------------------------------------------------
    UpgradeLink('EL_02', 'Eastleigh First Avenue upgrade', 'Eastleigh',
                1.4, 1.15, '[NaMSIP] widen 30→40 m; grade separation at Jogoo/Juja.'),
    UpgradeLink('EL_04', 'Mumias South Road — BRT corridor', 'Mumias',
                1.4, 1.15, '[NaMSIP] widen 30→40 m; listed BRT corridor.',
                relief=[('karen', 0.05)]),
    InterventionGroup('NRR_01', 'Nairobi Riverfront Road — full corridor',
                      RIVERFRONT_SEGMENTS,
                      '[NaMSIP] Quarry Rd ↔ Rabai Rd along Nairobi River, ~9 km, '
                      'LRT reserve; v1 segment anchors retained.'),

    # ---- NEW candidates from the plans (v2) ----------------------------------
    InterventionGroup('C2_NORTH', 'Circumferential Road C-2 northern section',
                      [UpgradeLink('C2_UP', 'Ring roads C-2 corridor upgrade',
                                   'Ring Road Ngara|Ring Road Kilimani|Mbagathi',
                                   1.5, 1.2, ''),
                       AddLink('C2_LINK', 'C-2 Kilimani — Mbagathi Way connector',
                               (36.7850, -1.2950), (36.7930, -1.3005), 1400, 11.1, 2, '')],
                      '[NIUPLAN] short-term priority ($12M) later dropped by KURA '
                      '(Njeru Table 9) — tests whether dropping it was justified.'),
    AddLink('VIAD_RC', 'Railway City CBD viaduct',
            (36.8180, -1.2930), (36.8280, -1.2980), 1000, 11.1, 4,
            '[NIUPLAN] Viaduct-1: Moi Ave–Enterprise, 4 lanes, removes rail-yard '
            'barrier between CBD and Industrial Area.'),
    InterventionGroup('RIV_BRIDGE', 'Quarry Rd & Bondo St river bridges',
                      [AddLink('BRG_QRY', 'Quarry Rd bridge over Nairobi River',
                               (36.8029, -1.2953), (36.8029, -1.3046), 700, 8.3, 2, ''),
                       AddLink('BRG_BND', 'Bondo Street bridge over Nairobi River',
                               (36.8165, -1.2919), (36.8165, -1.3010), 700, 8.3, 2, '')],
                      '[NaMSIP] two new motorized bridges connecting the northern '
                      'and southern spheres split by the river.'),
    AddLink('NILE_VIA', 'Nile Road viaduct into Industrial Area',
            (36.8420, -1.2960), (36.8460, -1.3060), 1300, 10.0, 2,
            '[NaMSIP] Jogoo/Nile extension viaduct; NaMSIP demand model expects '
            '15–20% redistribution onto it.'),
]

REGISTRY = {iv.id: iv for iv in INTERVENTIONS}
print(f"Defined {len(INTERVENTIONS)} candidate interventions:")
for iv in INTERVENTIONS:
    print(f"  {iv.id:<11} {iv.name}")


In [ ]:
# %SMOKE:skip
print("Running baseline (replicates=%d)..." % CONFIG['n_replicates_final'])
W_base, BASELINE_M = run_sim(scale=1.0)
BASELINE_ATT  = BASELINE_M['att_s']
BASELINE_COMP = BASELINE_M['completion_rate']

done, aborted, running = classify_vehicles(W_base)
print(f"A1 verification: completed={len(done):,} aborted={len(aborted):,} "
      f"incomplete={len(running):,}  (aborted no longer inflate ATT)")
print(f"Baseline ATT:        {BASELINE_ATT/60:.1f} min "
      f"(median {BASELINE_M['att_median_s']/60:.1f}, p90 {BASELINE_M['att_p90_s']/60:.1f})")
print(f"Baseline completion: {BASELINE_COMP*100:.1f}% | abort rate {BASELINE_M['abort_rate']*100:.1f}%")
print(f"Network speed:       {BASELINE_M['net_speed_kmh']:.1f} km/h")

with open(f"{CONFIG['results_dir']}/baseline.json", 'w') as f:
    json.dump({'metrics': BASELINE_M,
               'capacity_factor': CAPACITY_FACTOR_STATE['value'],
               'config': {k: v for k, v in CONFIG.items() if k != 'bbox'}}, f, indent=2)
print("Baseline saved")


In [ ]:
# %SMOKE:skip
def _eval_worker(payload):
    """Module-level so ProcessPoolExecutor can pickle it (D12).
    Each candidate gets its own CRN stream (A4) inside create_world."""
    applied_ids, cand_id = payload
    ivs = [REGISTRY[i] for i in applied_ids] + [REGISTRY[cand_id]]
    _, m = run_sim(interventions=ivs, replicates=CONFIG['n_replicates_search'])
    return cand_id, m['att_s'], m['completion_rate']


def evaluate_candidates(applied, remaining):
    payloads = [(tuple(iv.id for iv in applied), cand.id) for cand in remaining]
    results = []
    try:
        ctx = mp.get_context('fork')
        with ProcessPoolExecutor(max_workers=CONFIG['n_jobs'], mp_context=ctx) as pool:
            results = list(pool.map(_eval_worker, payloads))
    except Exception as e:
        print(f"(parallel eval unavailable: {e}; falling back to serial)")
        results = [_eval_worker(p) for p in payloads]
    return results


def greedy_search(candidates, baseline_att, min_improvement_pct=1.0, max_iterations=15):
    applied, remaining = [], list(candidates)
    current_att, history = baseline_att, []
    target_att = baseline_att * 0.75

    print("=" * 68)
    print("GREEDY SEARCH (CRN + parallel)")
    print(f"Baseline {baseline_att/60:.1f} min | target {target_att/60:.1f} min "
          f"| reps/candidate {CONFIG['n_replicates_search']} | jobs {CONFIG['n_jobs']}")
    print("=" * 68)

    for it in range(max_iterations):
        print(f"\nIteration {it+1} (ATT {current_att/60:.1f} min)")
        results = [(cid, (a if a == a else float('inf')), c)
                   for cid, a, c in evaluate_candidates(applied, remaining)]
        scored = sorted(results, key=lambda r: r[1])
        for cid, att, comp in scored:
            delta = current_att - att
            print(f"  {cid:<11} ATT={att/60:5.1f}min Δ={delta/60:+5.1f}min "
                  f"({delta/current_att*100:+5.1f}%) comp={comp*100:3.0f}%")
        best_id, best_att, _ = scored[0]
        best_delta = current_att - best_att
        impr = best_delta / current_att * 100
        if impr < min_improvement_pct:
            print(f"\nStopping — best improvement {impr:.1f}% < {min_improvement_pct}%")
            break

        best_iv = REGISTRY[best_id]
        applied.append(best_iv)
        remaining.remove(best_iv)
        current_att = best_att
        history.append({
            'iteration': it + 1, 'intervention_id': best_iv.id,
            'name': best_iv.name, 'att_minutes': current_att / 60,
            'improvement_pct': impr, 'rationale': best_iv.rationale, 'swap': False,
        })
        print(f"  ✓ selected [{best_iv.id}] {best_iv.name} -> {current_att/60:.1f} min")

        if (baseline_att - current_att) / baseline_att >= 0.25:
            print("\nTARGET REACHED (>=25% reduction)")
            break
        if not remaining:
            break
    return applied, remaining, current_att, history


def swap_test(applied, remaining, current_att, history, baseline_att, min_improvement_pct=1.0):
    """One local-refinement pass: try replacing each selected intervention with
    each unselected one; commit the first improving swap found per slot."""
    print("\n" + "=" * 68)
    print("SWAP TEST (one pass)")
    changed = True
    while changed:
        changed = False
        for idx in range(len(applied)):
            trial_base = [iv for j, iv in enumerate(applied) if j != idx]
            results = evaluate_candidates(trial_base, remaining)
            cur_id = applied[idx].id
            cur_best = (None, current_att)
            for cid, att, comp in results:
                if att < cur_best[1] and (current_att - att)/current_att*100 >= min_improvement_pct:
                    cur_best = (REGISTRY[cid], att)
            if cur_best[0] is not None:
                alt = cur_best[0]
                prev_att = current_att
                print(f"  swap [{cur_id}] -> [{alt.id}] "
                      f"({current_att/60:.1f} -> {cur_best[1]/60:.1f} min)")
                remaining.append(applied[idx])
                remaining.remove(alt)
                applied[idx] = alt
                current_att = cur_best[1]
                history.append({
                    'iteration': len(history)+1, 'intervention_id': alt.id,
                    'name': alt.name, 'att_minutes': current_att/60,
                    'improvement_pct': (prev_att - current_att) / prev_att * 100,
                    'rationale': alt.rationale, 'swap': True,
                })
                changed = True
                break
    return applied, remaining, current_att, history


applied, remaining, final_att, history = greedy_search(
    INTERVENTIONS, BASELINE_ATT, min_improvement_pct=1.0)
applied, remaining, final_att, history = swap_test(
    applied, remaining, final_att, history, BASELINE_ATT)


In [ ]:
# %SMOKE:skip
total_reduction = (BASELINE_ATT - final_att) / BASELINE_ATT * 100

print("\n" + "=" * 68)
print("RESULTS — Nairobi Road Network Intervention Analysis (methodology v2)")
print("=" * 68)
print(f"Baseline ATT:  {BASELINE_ATT/60:.1f} min "
      f"(p90 {BASELINE_M['att_p90_s']/60:.1f})")
print(f"Final ATT:     {final_att/60:.1f} min")
print(f"Reduction:     {(BASELINE_ATT-final_att)/60:.1f} min ({total_reduction:.1f}%)")
print(f"Target met:    {'YES' if total_reduction >= 25 else 'NO'}\n")

print(f"Selected portfolio ({len(applied)}):")
for i, iv in enumerate(applied, 1):
    print(f"  {i}. [{iv.id}] {iv.name}")
    print(f"     {iv.rationale}")

# Final configuration with full replications for headline numbers
_, FINAL_M = run_sim(interventions=applied, scale=1.0)
print(f"\nFinal (reps={FINAL_M['n_replicates']}): ATT={FINAL_M['att_s']/60:.1f} min | "
      f"completion={FINAL_M['completion_rate']*100:.1f}% | "
      f"speed={FINAL_M['net_speed_kmh']:.1f} km/h")

results = {
    'methodology': 'v2 — CRN, replications, aborted-trip exclusion, '
                   'capacity-factor calibration to NIUPLAN 20 km/h',
    'baseline': BASELINE_M,
    'final': FINAL_M,
    'total_reduction_pct': total_reduction,
    'target_met': bool(total_reduction >= 25),
    'capacity_factor': CAPACITY_FACTOR_STATE['value'],
    'applied_interventions': [
        {'id': iv.id, 'name': iv.name, 'rationale': iv.rationale} for iv in applied],
    'greedy_history': history,
    'sources': ['NIUPLAN (JICA 2014)', 'NaMSIP Eastlands Urban Renewal Plan Vol.2 (2019)',
                'Njeru 2024 — NIUPLAN implementation gaps (UoN MA thesis)'],
}
with open(f"{CONFIG['results_dir']}/results.json", 'w') as f:
    json.dump(results, f, indent=2)
print("Results saved")

if history:
    atts = [BASELINE_ATT/60] + [h['att_minutes'] for h in history]
    labels = ['Base'] + [h['intervention_id'] for h in history]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    ax = axes[0]
    ax.plot(range(len(atts)), atts, marker='o', linewidth=2)
    ax.axhline(BASELINE_ATT*0.75/60, color='green', linestyle='--', label='25% target')
    ax.set_xticks(range(len(labels)), labels, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('ATT (min)'), ax.legend(), ax.set_title('Cumulative path')
    ax = axes[1]
    ax.bar(range(len(history)),
           [h['improvement_pct'] for h in history])
    ax.set_xticks(range(len(history)),
                  [h['intervention_id'] for h in history],
                  rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Marginal ATT reduction (%)'), ax.set_title('Marginal gains')
    plt.tight_layout()
    plt.show()
